In [0]:
import requests
import time

def fetch_with_retry(url, max_attempts=3, base_delay=1.0, timeout=30):
    """
    Fetch a URL with exponential-backoff retry.

    Retries ONLY on transient failures (connection errors, timeouts, 5xx).
    Fails fast on permanent errors (4xx) — retrying a 404 never helps.
    Returns the successful Response, or raises after exhausting attempts.
    """
    for attempt in range(1, max_attempts + 1):
        try:
            resp = requests.get(url, timeout=timeout)

            if 400 <= resp.status_code < 500:
                # permanent client error — do NOT retry
                raise RuntimeError(f"Permanent HTTP {resp.status_code} for {url} (not retrying)")

            if resp.status_code >= 500:
                # transient server error — allow retry
                raise ConnectionError(f"Server HTTP {resp.status_code} for {url}")

            resp.raise_for_status()
            print(f"[fetch_with_retry] SUCCESS on attempt {attempt}: {url}")
            return resp

        except RuntimeError:
            # permanent — bubble up immediately, no retry
            raise

        except (requests.ConnectionError, requests.Timeout, ConnectionError) as e:
            if attempt < max_attempts:
                delay = base_delay * (2 ** (attempt - 1))   # 1s, 2s, 4s...
                print(f"[fetch_with_retry] attempt {attempt}/{max_attempts} failed: {e}")
                print(f"[fetch_with_retry] backing off {delay:.0f}s before retry")
                time.sleep(delay)
            else:
                print(f"[fetch_with_retry] exhausted {max_attempts} attempts for {url}")
                raise

In [0]:
#THE HEALING LOG

CATALOG = "airbnb_obs"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.monitoring.healing_log (
    heal_ts         TIMESTAMP,
    table_name      STRING,
    action          STRING,      -- e.g. 'quarantine_orphans'
    records_healed  BIGINT,      -- how many rows acted on
    details         STRING
) USING DELTA
""")
print("✔ healing_log ready")

In [0]:
# THE REMEDIATION FUNCTION

from pyspark.sql import functions as F
from datetime import datetime, timezone

def heal_orphan_bookings():
    fact = spark.table(f"{CATALOG}.gold.fact_bookings")
    dims = spark.table(f"{CATALOG}.gold.dim_listings").select("listing_id")

    # orphans = fact rows with no matching listing_id in the dimension
    orphans = fact.join(dims, on="listing_id", how="left_anti")
    n = orphans.count()

    if n == 0:
        print(" no orphans : nothing to heal")
        return

    # 1. preserve the bad rows (never delete silently)
    (orphans.write.mode("append")
            .saveAsTable(f"{CATALOG}.gold.fact_bookings_quarantine"))

    # 2. rewrite fact with only the clean rows
    clean = fact.join(dims, on="listing_id", how="left_semi")
    (clean.write.mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(f"{CATALOG}.gold.fact_bookings"))

    # 3. log the heal
    (spark.createDataFrame(
        [(datetime.now(timezone.utc), "fact_bookings", "quarantine_orphans",
          int(n), f"moved {n} orphan booking(s) to fact_bookings_quarantine")],
        ["heal_ts","table_name","action","records_healed","details"])
      .write.mode("append")
      .saveAsTable(f"{CATALOG}.monitoring.healing_log"))

    print(f"⚠ healed {n} orphan(s) — quarantined and logged")

heal_orphan_bookings()

In [0]:
#TEST IT: LET INSERT BROKEN DATA

from datetime import datetime, timezone

# inject ONE orphan: a booking pointing at a listing_id that doesn't exist in dim_listings
orphan_row = spark.createDataFrame(
    [("ORPHAN_TEST", 999999, "2026-01-01", 3, 300.0, 50.0, 30.0, "confirmed")],
    ["booking_id","listing_id","booking_date","nights_booked",
     "booking_amount","cleaning_fee","service_fee","booking_status"]
)

# align to the real fact schema, then append
fact_cols = spark.table(f"{CATALOG}.gold.fact_bookings").columns
orphan_row = orphan_row.select(*[c for c in fact_cols if c in orphan_row.columns])
orphan_row.write.mode("append").saveAsTable(f"{CATALOG}.gold.fact_bookings")

print("injected 1 orphan (listing_id 999999)")

In [0]:
#VERDICT
spark.table(f"{CATALOG}.gold.fact_bookings").printSchema()

In [0]:
display(spark.table(f"{CATALOG}.monitoring.healing_log"))

In [0]:
#CHECK THE FIX

CATALOG = "airbnb_obs"

# 1. fact_bookings should be back to 5000 (orphan removed)
print("fact_bookings:", spark.table(f"{CATALOG}.gold.fact_bookings").count())

# 2. the orphan should be sitting in quarantine
display(spark.table(f"{CATALOG}.gold.fact_bookings_quarantine"))

# 3. re-run the heal — should now say "nothing to heal" (proves idempotency)
heal_orphan_bookings()

In [0]:
heal_orphan_bookings()